# Aggressive 2048 Rainbow-Style Notebook for Very High Mean Score

Notebook này là một bản **mạnh hơn đáng kể** so với v4, theo hướng nhắm tới hiệu năng rất cao trên 2048.

## Thành phần chính
- OpenSpiel 2048
- Strategic observation wrapper: **21 x 4 x 4**
- **Distributional DQN (C51)**
- **Double DQN**
- **Dueling**
- **NoisyNet**
- **Prioritized Replay**
- **n-step returns**
- **Illegal-action auxiliary loss**
- Probe eval + best checkpoint + resume

## Ghi chú
Mục tiêu mean cực lớn như **10000+** đã là khó; **100000 mean** là mục tiêu rất cực đoan và thường không thực tế với pure value-based RL baseline. Notebook này đi theo hướng mạnh nhất trong khung DQN/Rainbow-lite để nhắm score cao hơn hẳn v4.

In [ ]:

# Uncomment if your environment is missing dependencies.
# !pip -q install open_spiel torch numpy matplotlib

In [ ]:

from __future__ import annotations

import os
import json
import time
import math
import random
import shutil
from dataclasses import dataclass, asdict
from collections import deque, Counter, namedtuple
from pathlib import Path
from typing import Deque, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

try:
    import pyspiel
except ImportError as e:
    raise ImportError("Please install open_spiel / pyspiel first.") from e

In [ ]:

@dataclass
class Config:
    game_name: str = "2048"
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    workdir: str = "./aggressive_2048_rainbow"
    run_name: str = "rainbow_high_mean"

    num_episodes: int = 20000
    max_steps_per_episode: int = 4000
    learn_start: int = 20000
    learn_every: int = 4
    gradient_steps_per_update: int = 1
    target_sync_every: int = 4000

    buffer_size: int = 500_000
    batch_size: int = 256
    per_alpha: float = 0.6
    per_beta_start: float = 0.4
    per_beta_frames: int = 2_500_000
    per_eps: float = 1e-5

    gamma: float = 0.997
    n_step: int = 5
    lr: float = 1e-4
    lr_finetune: float = 5e-5
    grad_clip: float = 10.0
    weight_decay: float = 0.0

    num_atoms: int = 51
    v_min: float = -10.0
    v_max: float = 20000.0

    eps_start: float = 0.20
    eps_end: float = 0.01
    eps_decay_steps: int = 1_200_000

    reward_mode: str = "strategic_shaped_plus"
    w_raw_score: float = 1.0
    w_empty_delta: float = 2.0
    w_monotonicity: float = 1.0
    w_smoothness: float = 0.3
    w_corner_max: float = 4.0
    w_new_max_tile: float = 8.0
    w_merge_opportunity: float = 1.0
    w_dead_board_penalty: float = 3.0

    illegal_aux_weight: float = 0.10
    illegal_margin: float = 0.50

    probe_every_episodes: int = 50
    probe_num_seeds: int = 15
    probe_seed_start: int = 20_000
    final_eval_seeds: int = 50
    final_eval_seed_start: int = 30_000

    checkpoint_every_episodes: int = 100
    heartbeat_every_episodes: int = 10
    keep_last_n_checkpoints: int = 5
    save_replay_in_checkpoint: bool = False
    resume_from: str = "latest"

    best_primary_metric: str = "mean_return"
    best_secondary_metric: str = "p512_rate"
    finetune_after_episode: int = 12000

CFG = Config()
os.makedirs(CFG.workdir, exist_ok=True)
print(asdict(CFG))

In [ ]:

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)
DEVICE = torch.device(CFG.device)
DEVICE

In [ ]:

def create_game_and_state(seed: Optional[int] = None):
    game = pyspiel.load_game(CFG.game_name)
    state = game.new_initial_state()
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
    return game, state

def legal_actions_mask(num_actions: int, legal_actions: List[int]) -> np.ndarray:
    mask = np.zeros(num_actions, dtype=np.float32)
    mask[legal_actions] = 1.0
    return mask

def board_from_observation_tensor(obs_tensor: np.ndarray) -> np.ndarray:
    return np.asarray(obs_tensor, dtype=np.float32).reshape(4, 4).copy()

def log2_tile(x: float) -> int:
    x = int(x)
    if x <= 0:
        return 0
    return int(round(math.log2(x)))

def empty_count(board: np.ndarray) -> int:
    return int(np.sum(board == 0))

def max_tile(board: np.ndarray) -> int:
    return int(board.max())

def monotonicity_score(board: np.ndarray) -> float:
    b = np.where(board > 0, np.log2(board, where=board>0, out=np.zeros_like(board, dtype=np.float32)), 0.0)
    score = 0.0
    for row in b:
        inc = np.sum(np.maximum(0.0, row[:-1] - row[1:]))
        dec = np.sum(np.maximum(0.0, row[1:] - row[:-1]))
        score += max(inc, dec)
    for col in b.T:
        inc = np.sum(np.maximum(0.0, col[:-1] - col[1:]))
        dec = np.sum(np.maximum(0.0, col[1:] - col[:-1]))
        score += max(inc, dec)
    return float(score)

def smoothness_score(board: np.ndarray) -> float:
    b = np.where(board > 0, np.log2(board, where=board>0, out=np.zeros_like(board, dtype=np.float32)), 0.0)
    total = 0.0
    for r in range(4):
        for c in range(4):
            if r + 1 < 4:
                total += abs(float(b[r, c] - b[r + 1, c]))
            if c + 1 < 4:
                total += abs(float(b[r, c] - b[r, c + 1]))
    return float(total)

def corner_max_flag(board: np.ndarray) -> float:
    mx = board.max()
    corners = [board[0,0], board[0,3], board[3,0], board[3,3]]
    return 1.0 if mx in corners and mx > 0 else 0.0

def merge_opportunities(board: np.ndarray) -> int:
    count = 0
    for r in range(4):
        for c in range(4):
            if board[r, c] == 0:
                continue
            if r + 1 < 4 and board[r, c] == board[r + 1, c]:
                count += 1
            if c + 1 < 4 and board[r, c] == board[r, c + 1]:
                count += 1
    return count

def strategic_planes(board: np.ndarray) -> np.ndarray:
    planes = np.zeros((21, 4, 4), dtype=np.float32)
    for r in range(4):
        for c in range(4):
            k = max(0, min(15, log2_tile(board[r, c])))
            planes[k, r, c] = 1.0
    planes[16] = empty_count(board) / 16.0
    planes[17] = min(log2_tile(max_tile(board)) / 15.0, 1.0)
    planes[18] = monotonicity_score(board) / 32.0
    planes[19] = min(smoothness_score(board) / 64.0, 1.0)
    planes[20] = corner_max_flag(board)
    return planes

def raw_score_delta(prev_return: float, new_return: float) -> float:
    return float(new_return - prev_return)

def shaped_reward(prev_board: np.ndarray, board: np.ndarray, prev_return: float, new_return: float, done: bool) -> float:
    base = CFG.w_raw_score * raw_score_delta(prev_return, new_return)
    if CFG.reward_mode == "raw":
        return base

    e_prev = empty_count(prev_board)
    e_curr = empty_count(board)
    m_prev = monotonicity_score(prev_board)
    m_curr = monotonicity_score(board)
    s_prev = smoothness_score(prev_board)
    s_curr = smoothness_score(board)
    c_prev = corner_max_flag(prev_board)
    c_curr = corner_max_flag(board)
    mx_prev = max_tile(prev_board)
    mx_curr = max_tile(board)
    mo_prev = merge_opportunities(prev_board)
    mo_curr = merge_opportunities(board)

    reward = base
    reward += CFG.w_empty_delta * (e_curr - e_prev)
    reward += CFG.w_monotonicity * (m_curr - m_prev)
    reward -= CFG.w_smoothness * max(0.0, s_curr - s_prev)
    reward += CFG.w_corner_max * max(0.0, c_curr - c_prev)
    reward += CFG.w_new_max_tile * (1.0 if mx_curr > mx_prev else 0.0)
    reward += CFG.w_merge_opportunity * (mo_curr - mo_prev)
    if CFG.reward_mode == "strategic_shaped_plus" and done and empty_count(board) == 0:
        reward -= CFG.w_dead_board_penalty
    return float(reward)

In [ ]:

class NoisyLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, sigma0: float = 0.5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.empty(out_features, in_features))
        self.register_buffer("weight_eps", torch.empty(out_features, in_features))
        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))
        self.register_buffer("bias_eps", torch.empty(out_features))
        self.sigma0 = sigma0
        self.reset_parameters()
        self.reset_noise()

    def reset_parameters(self):
        mu_range = 1 / math.sqrt(self.in_features)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.weight_sigma.data.fill_(self.sigma0 / math.sqrt(self.in_features))
        self.bias_mu.data.uniform_(-mu_range, mu_range)
        self.bias_sigma.data.fill_(self.sigma0 / math.sqrt(self.out_features))

    def _scale_noise(self, size: int) -> torch.Tensor:
        x = torch.randn(size, device=self.weight_mu.device)
        return x.sign() * x.abs().sqrt()

    def reset_noise(self):
        eps_in = self._scale_noise(self.in_features)
        eps_out = self._scale_noise(self.out_features)
        self.weight_eps.copy_(eps_out.outer(eps_in))
        self.bias_eps.copy_(eps_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.training:
            weight = self.weight_mu + self.weight_sigma * self.weight_eps
            bias = self.bias_mu + self.bias_sigma * self.bias_eps
        else:
            weight = self.weight_mu
            bias = self.bias_mu
        return F.linear(x, weight, bias)

class Rainbow2048Net(nn.Module):
    def __init__(self, in_channels: int, num_actions: int, num_atoms: int):
        super().__init__()
        self.num_actions = num_actions
        self.num_atoms = num_atoms
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=2, stride=1), nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=2, stride=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=2, stride=1), nn.ReLU(),
        )
        self.val_fc1 = NoisyLinear(128, 256)
        self.val_fc2 = NoisyLinear(256, num_atoms)
        self.adv_fc1 = NoisyLinear(128, 256)
        self.adv_fc2 = NoisyLinear(256, num_actions * num_atoms)
        self.illegal_head = nn.Sequential(
            nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, num_actions)
        )

    def reset_noise(self):
        for m in self.modules():
            if isinstance(m, NoisyLinear):
                m.reset_noise()

    def forward(self, x: torch.Tensor):
        h = self.conv(x).flatten(1)
        v = F.relu(self.val_fc1(h))
        v = self.val_fc2(v).view(-1, 1, self.num_atoms)
        a = F.relu(self.adv_fc1(h))
        a = self.adv_fc2(a).view(-1, self.num_actions, self.num_atoms)
        logits = v + a - a.mean(dim=1, keepdim=True)
        illegal_logits = self.illegal_head(h)
        return logits, illegal_logits

    def dist(self, x: torch.Tensor):
        logits, illegal_logits = self.forward(x)
        probs = F.softmax(logits, dim=-1).clamp_min(1e-6)
        probs = probs / probs.sum(dim=-1, keepdim=True)
        return probs, illegal_logits

    def q_values(self, x: torch.Tensor, support: torch.Tensor):
        probs, illegal_logits = self.dist(x)
        q = torch.sum(probs * support.view(1, 1, -1), dim=-1)
        return q, illegal_logits

In [ ]:

Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "done", "legal_mask", "next_legal_mask"])

class SumTree:
    def __init__(self, capacity: int):
        self.capacity = int(capacity)
        self.tree = np.zeros(2 * capacity, dtype=np.float32)
        self.data = [None] * capacity
        self.ptr = 0
        self.size = 0

    def add(self, priority: float, data):
        idx = self.ptr + self.capacity
        self.data[self.ptr] = data
        self.update(idx, priority)
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def update(self, idx: int, priority: float):
        change = priority - self.tree[idx]
        self.tree[idx] = priority
        idx //= 2
        while idx >= 1:
            self.tree[idx] += change
            idx //= 2

    def get(self, s: float):
        idx = 1
        while idx < self.capacity:
            left = 2 * idx
            if s <= self.tree[left]:
                idx = left
            else:
                s -= self.tree[left]
                idx = left + 1
        data_idx = idx - self.capacity
        return idx, self.tree[idx], self.data[data_idx]

    @property
    def total(self):
        return float(self.tree[1])

class PrioritizedNStepReplay:
    def __init__(self, capacity: int, alpha: float, gamma: float, n_step: int, per_eps: float):
        self.capacity = capacity
        self.alpha = alpha
        self.gamma = gamma
        self.n_step = n_step
        self.per_eps = per_eps
        self.tree = SumTree(capacity)
        self.max_priority = 1.0
        self.nstep_buffer = deque(maxlen=n_step)

    def __len__(self):
        return self.tree.size

    def _get_n_step_info(self):
        reward = 0.0
        next_state = self.nstep_buffer[-1].next_state
        done = self.nstep_buffer[-1].done
        next_legal_mask = self.nstep_buffer[-1].next_legal_mask
        for i, tr in enumerate(self.nstep_buffer):
            reward += (self.gamma ** i) * tr.reward
            if tr.done:
                next_state = tr.next_state
                done = True
                next_legal_mask = tr.next_legal_mask
                break
        first = self.nstep_buffer[0]
        return Transition(first.state, first.action, reward, next_state, done, first.legal_mask, next_legal_mask)

    def add(self, tr: Transition):
        self.nstep_buffer.append(tr)
        if len(self.nstep_buffer) < self.n_step and not tr.done:
            return
        ntr = self._get_n_step_info()
        priority = (self.max_priority + self.per_eps) ** self.alpha
        self.tree.add(priority, ntr)
        if tr.done:
            while len(self.nstep_buffer) > 1:
                self.nstep_buffer.popleft()
                ntr = self._get_n_step_info()
                self.tree.add(priority, ntr)
            self.nstep_buffer.clear()

    def sample(self, batch_size: int, beta: float):
        batch, idxs, priorities = [], [], []
        segment = self.tree.total / batch_size
        for i in range(batch_size):
            s = random.uniform(segment * i, segment * (i + 1))
            idx, p, data = self.tree.get(s)
            batch.append(data)
            idxs.append(idx)
            priorities.append(p)
        probs = np.asarray(priorities, dtype=np.float32) / max(self.tree.total, 1e-8)
        weights = (self.tree.size * probs) ** (-beta)
        weights /= weights.max() + 1e-8
        return batch, np.asarray(idxs, dtype=np.int64), weights.astype(np.float32)

    def update_priorities(self, idxs, priorities):
        for idx, p in zip(idxs, priorities):
            p = float(max(p, self.per_eps))
            self.tree.update(int(idx), p ** self.alpha)
            self.max_priority = max(self.max_priority, p)

    def state_dict(self):
        return {
            "capacity": self.capacity,
            "alpha": self.alpha,
            "gamma": self.gamma,
            "n_step": self.n_step,
            "per_eps": self.per_eps,
            "tree_tree": self.tree.tree.copy(),
            "tree_data": self.tree.data.copy(),
            "tree_ptr": self.tree.ptr,
            "tree_size": self.tree.size,
            "max_priority": self.max_priority,
        }

    def load_state_dict(self, sd):
        self.capacity = sd["capacity"]
        self.alpha = sd["alpha"]
        self.gamma = sd["gamma"]
        self.n_step = sd["n_step"]
        self.per_eps = sd["per_eps"]
        self.tree = SumTree(self.capacity)
        self.tree.tree = sd["tree_tree"].copy()
        self.tree.data = sd["tree_data"].copy()
        self.tree.ptr = sd["tree_ptr"]
        self.tree.size = sd["tree_size"]
        self.max_priority = sd["max_priority"]

In [ ]:

class RainbowAgent:
    def __init__(self, obs_shape, num_actions: int, cfg: Config):
        self.cfg = cfg
        self.num_actions = num_actions
        self.support = torch.linspace(cfg.v_min, cfg.v_max, cfg.num_atoms, device=DEVICE)
        self.delta_z = (cfg.v_max - cfg.v_min) / (cfg.num_atoms - 1)
        self.online = Rainbow2048Net(obs_shape[0], num_actions, cfg.num_atoms).to(DEVICE)
        self.target = Rainbow2048Net(obs_shape[0], num_actions, cfg.num_atoms).to(DEVICE)
        self.target.load_state_dict(self.online.state_dict())
        self.target.eval()
        self.optimizer = Adam(self.online.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        self.global_step = 0
        self.beta = cfg.per_beta_start

    def current_epsilon(self):
        t = min(self.global_step, self.cfg.eps_decay_steps)
        frac = t / max(1, self.cfg.eps_decay_steps)
        return self.cfg.eps_start + frac * (self.cfg.eps_end - self.cfg.eps_start)

    def reset_noise(self):
        self.online.reset_noise()
        self.target.reset_noise()

    @torch.no_grad()
    def act(self, state_planes, legal_mask, greedy=False):
        x = torch.tensor(state_planes, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        q, illegal_logits = self.online.q_values(x, self.support)
        q = q[0]
        raw_argmax = int(torch.argmax(q).item())
        raw_argmax_is_legal = bool(legal_mask[raw_argmax] > 0.5)

        masked_q = q.clone()
        masked_q[torch.tensor(legal_mask < 0.5, dtype=torch.bool, device=DEVICE)] = -1e9

        if greedy:
            action = int(torch.argmax(masked_q).item())
        else:
            if random.random() < self.current_epsilon():
                legal_actions = np.flatnonzero(legal_mask > 0.5)
                action = int(np.random.choice(legal_actions))
            else:
                action = int(torch.argmax(masked_q).item())

        return action, {
            "raw_argmax": raw_argmax,
            "raw_argmax_is_legal": raw_argmax_is_legal,
            "q_values": q.detach().cpu().numpy(),
            "illegal_logits": illegal_logits[0].detach().cpu().numpy(),
        }

    def maybe_finetune_lr(self, episode_idx):
        if episode_idx == self.cfg.finetune_after_episode:
            for g in self.optimizer.param_groups:
                g["lr"] = self.cfg.lr_finetune

    def projection_distribution(self, next_dist, rewards, dones):
        batch = rewards.shape[0]
        tz = rewards.unsqueeze(1) + (1.0 - dones.unsqueeze(1)) * (self.cfg.gamma ** self.cfg.n_step) * self.support.unsqueeze(0)
        tz = tz.clamp(self.cfg.v_min, self.cfg.v_max)
        b = (tz - self.cfg.v_min) / self.delta_z
        l = b.floor().long()
        u = b.ceil().long()
        proj = torch.zeros((batch, self.cfg.num_atoms), device=DEVICE)
        offset = torch.linspace(0, (batch - 1) * self.cfg.num_atoms, batch, device=DEVICE).long().unsqueeze(1)
        proj.view(-1).index_add_(0, (l + offset).view(-1), (next_dist * (u.float() - b)).view(-1))
        proj.view(-1).index_add_(0, (u + offset).view(-1), (next_dist * (b - l.float())).view(-1))
        return proj

    def learn(self, replay):
        if len(replay) < max(self.cfg.batch_size, self.cfg.learn_start):
            return None

        beta_frac = min(1.0, self.global_step / max(1, self.cfg.per_beta_frames))
        self.beta = self.cfg.per_beta_start + beta_frac * (1.0 - self.cfg.per_beta_start)

        batch, idxs, weights = replay.sample(self.cfg.batch_size, self.beta)
        states = torch.tensor(np.stack([b.state for b in batch]), dtype=torch.float32, device=DEVICE)
        actions = torch.tensor([b.action for b in batch], dtype=torch.long, device=DEVICE)
        rewards = torch.tensor([b.reward for b in batch], dtype=torch.float32, device=DEVICE)
        next_states = torch.tensor(np.stack([b.next_state for b in batch]), dtype=torch.float32, device=DEVICE)
        dones = torch.tensor([float(b.done) for b in batch], dtype=torch.float32, device=DEVICE)
        legal_masks = torch.tensor(np.stack([b.legal_mask for b in batch]), dtype=torch.float32, device=DEVICE)
        next_legal_masks = torch.tensor(np.stack([b.next_legal_mask for b in batch]), dtype=torch.float32, device=DEVICE)
        weights_t = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

        self.online.train()
        self.reset_noise()

        dist, _ = self.online.dist(states)
        chosen_dist = dist[torch.arange(self.cfg.batch_size, device=DEVICE), actions]

        with torch.no_grad():
            next_q_online, _ = self.online.q_values(next_states, self.support)
            next_q_online = next_q_online.masked_fill(next_legal_masks < 0.5, -1e9)
            next_actions = torch.argmax(next_q_online, dim=1)

            next_dist_target, _ = self.target.dist(next_states)
            next_action_dist = next_dist_target[torch.arange(self.cfg.batch_size, device=DEVICE), next_actions]
            target_proj = self.projection_distribution(next_action_dist, rewards, dones)

        per_sample_rl = -(target_proj * torch.log(chosen_dist.clamp_min(1e-6))).sum(dim=1)

        q_online, _ = self.online.q_values(states, self.support)
        legal_q = q_online.masked_fill(legal_masks < 0.5, -1e9).max(dim=1).values
        illegal_q = q_online.masked_fill(legal_masks > 0.5, -1e9).max(dim=1).values
        no_illegal = (legal_masks.sum(dim=1) == legal_masks.shape[1])
        illegal_aux = torch.where(
            no_illegal,
            torch.zeros_like(legal_q),
            F.relu(self.cfg.illegal_margin + illegal_q - legal_q)
        )

        loss = (weights_t * (per_sample_rl + self.cfg.illegal_aux_weight * illegal_aux)).mean()

        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), self.cfg.grad_clip)
        self.optimizer.step()

        prios = per_sample_rl.detach().cpu().numpy() + self.cfg.illegal_aux_weight * illegal_aux.detach().cpu().numpy()
        replay.update_priorities(idxs, prios + self.cfg.per_eps)

        if self.global_step % self.cfg.target_sync_every == 0:
            self.target.load_state_dict(self.online.state_dict())

        return {
            "loss": float(loss.item()),
            "rl_loss": float(per_sample_rl.mean().item()),
            "illegal_aux": float(illegal_aux.mean().item()),
            "beta": float(self.beta),
        }

In [ ]:

def reset_env(seed=None):
    game, state = create_game_and_state(seed)
    return game, state, game.num_distinct_actions()

def observe_state(state, num_actions):
    obs = np.array(state.observation_tensor(), dtype=np.float32)
    board = board_from_observation_tensor(obs)
    planes = strategic_planes(board)
    legal = state.legal_actions()
    lmask = legal_actions_mask(num_actions, legal)
    return board, planes, legal, lmask

def step_env(state, action):
    state.apply_action(action)

def eval_policy(agent, num_seeds, seed_start):
    rows = []
    max_tile_counter = Counter()
    illegal_pref_count = 0
    total_decisions = 0

    for i in range(num_seeds):
        seed = seed_start + i
        game, state, num_actions = reset_env(seed)
        while not state.is_terminal():
            board, planes, legal, lmask = observe_state(state, num_actions)
            action, info = agent.act(planes, lmask, greedy=True)
            illegal_pref_count += int(not info["raw_argmax_is_legal"])
            total_decisions += 1
            step_env(state, action)

        board, planes, legal, lmask = observe_state(state, num_actions)
        final_return = float(state.returns()[0]) if hasattr(state, "returns") else 0.0
        tile = max_tile(board)
        max_tile_counter[tile] += 1
        rows.append({"seed": seed, "score": final_return, "max_tile": tile})

    for row in rows:
        print(f"eval_seed={row['seed']} | score={row['score']:.1f} | max_tile={row['max_tile']}")

    scores = np.array([r["score"] for r in rows], dtype=np.float32)
    tiles = np.array([r["max_tile"] for r in rows], dtype=np.float32)
    summary = {
        "mean_return": float(scores.mean()),
        "std_return": float(scores.std()),
        "median_return": float(np.median(scores)),
        "mean_max_tile": float(tiles.mean()),
        "max_tile_distribution": dict(sorted(max_tile_counter.items())),
        "fraction_illegal_actions_avoided": float(1.0 - illegal_pref_count / max(1, total_decisions)),
        "p512_rate": float(np.mean(tiles >= 512)),
        "p1024_rate": float(np.mean(tiles >= 1024)),
        "num_eval_seeds": int(num_seeds),
        "rows": rows,
    }
    print("Mean greedy return:", summary["mean_return"])
    print("Std greedy return:", summary["std_return"])
    print("Median greedy return:", summary["median_return"])
    print("Mean max tile:", summary["mean_max_tile"])
    print("Max tile distribution:", summary["max_tile_distribution"])
    print("Fraction of illegal actions avoided:", summary["fraction_illegal_actions_avoided"])
    print("P(max_tile >= 512):", summary["p512_rate"])
    print("P(max_tile >= 1024):", summary["p1024_rate"])
    print("Number of eval seeds:", summary["num_eval_seeds"])
    return summary

In [ ]:

def metric_tuple(summary, primary, secondary):
    return (summary.get(primary, float("-inf")), summary.get(secondary, float("-inf")))

class CheckpointManager:
    def __init__(self, cfg):
        self.cfg = cfg
        self.root = Path(cfg.workdir)
        self.ckpt_dir = self.root / "checkpoints"
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.status_path = self.root / "status.json"

    def save(self, tag, payload):
        path = self.ckpt_dir / f"{tag}.pt"
        torch.save(payload, path)
        latest = self.ckpt_dir / "latest.pt"
        shutil.copy2(path, latest)
        with open(self.status_path, "w", encoding="utf-8") as f:
            json.dump({"latest_checkpoint": str(latest), "tag": tag, "saved_at": time.time()}, f, indent=2)
        return path

    def resolve(self, which):
        if which in (None, "", "none"):
            return None
        if which == "latest":
            p = self.ckpt_dir / "latest.pt"
            return p if p.exists() else None
        if which == "best_mean":
            p = self.ckpt_dir / "best_mean.pt"
            return p if p.exists() else None
        p = Path(which)
        return p if p.exists() else None

In [ ]:

def train(cfg):
    set_seed(cfg.seed)
    manager = CheckpointManager(cfg)
    game, state, num_actions = reset_env(cfg.seed)
    _, planes0, _, _ = observe_state(state, num_actions)

    agent = RainbowAgent(tuple(planes0.shape), num_actions, cfg)
    replay = PrioritizedNStepReplay(cfg.buffer_size, cfg.per_alpha, cfg.gamma, cfg.n_step, cfg.per_eps)

    start_episode = 1
    best_summary = None
    best_metric = (float("-inf"), float("-inf"))
    history = {"episode": [], "score": [], "max_tile": [], "loss": [], "probe_mean_return": [], "probe_p512": []}

    resume_path = manager.resolve(cfg.resume_from)
    if resume_path is not None:
        print(f"Resuming from {resume_path}")
        ckpt = torch.load(resume_path, map_location=DEVICE)
        agent.online.load_state_dict(ckpt["online"])
        agent.target.load_state_dict(ckpt["target"])
        agent.optimizer.load_state_dict(ckpt["optimizer"])
        agent.global_step = ckpt.get("global_step", 0)
        start_episode = ckpt["episode"] + 1
        history = ckpt.get("history", history)
        best_summary = ckpt.get("best_summary", None)
        best_metric = tuple(ckpt.get("best_metric", best_metric))
        if cfg.save_replay_in_checkpoint and "replay" in ckpt:
            replay.load_state_dict(ckpt["replay"])

    t0 = time.time()
    score_window = deque(maxlen=20)
    tile_window = deque(maxlen=20)
    recent_losses = deque(maxlen=100)

    for episode in range(start_episode, cfg.num_episodes + 1):
        agent.maybe_finetune_lr(episode)
        game, state, num_actions = reset_env(cfg.seed + episode)
        board, planes, legal, lmask = observe_state(state, num_actions)
        prev_return = float(state.returns()[0]) if hasattr(state, "returns") else 0.0

        ep_score = 0.0
        ep_steps = 0

        while not state.is_terminal() and ep_steps < cfg.max_steps_per_episode:
            action, info = agent.act(planes, lmask, greedy=False)
            prev_board = board.copy()
            step_env(state, action)
            new_return = float(state.returns()[0]) if hasattr(state, "returns") else prev_return
            next_board, next_planes, next_legal, next_lmask = observe_state(state, num_actions)
            done = state.is_terminal()
            reward = shaped_reward(prev_board, next_board, prev_return, new_return, done)

            replay.add(Transition(planes, action, reward, next_planes, done, lmask, next_lmask))

            planes = next_planes
            board = next_board
            lmask = next_lmask
            prev_return = new_return
            ep_score = new_return
            ep_steps += 1
            agent.global_step += 1

            if agent.global_step >= cfg.learn_start and agent.global_step % cfg.learn_every == 0:
                for _ in range(cfg.gradient_steps_per_update):
                    out = agent.learn(replay)
                    if out is not None:
                        recent_losses.append(out["loss"])

        ep_tile = max_tile(board)
        score_window.append(ep_score)
        tile_window.append(ep_tile)
        history["episode"].append(episode)
        history["score"].append(ep_score)
        history["max_tile"].append(ep_tile)
        history["loss"].append(float(np.mean(recent_losses)) if recent_losses else np.nan)

        if episode % cfg.heartbeat_every_episodes == 0:
            print(
                f"[heartbeat] ep={episode}/{cfg.num_episodes} | "
                f"global_step={agent.global_step} | eps={agent.current_epsilon():.4f} | "
                f"replay={len(replay)} | score_ma20={np.mean(score_window):.1f} | "
                f"tile_ma20={np.mean(tile_window):.1f} | "
                f"loss_ma100={np.mean(recent_losses) if recent_losses else float('nan'):.4f} | "
                f"elapsed_min={(time.time() - t0)/60.0:.1f}"
            )

        if episode % cfg.probe_every_episodes == 0:
            print(f"\n[probe] episode={episode}")
            probe = eval_policy(agent, cfg.probe_num_seeds, cfg.probe_seed_start)
            history["probe_mean_return"].append((episode, probe["mean_return"]))
            history["probe_p512"].append((episode, probe["p512_rate"]))
            m = metric_tuple(probe, cfg.best_primary_metric, cfg.best_secondary_metric)
            if m > best_metric:
                best_metric = m
                best_summary = probe
                payload = {
                    "episode": episode,
                    "global_step": agent.global_step,
                    "online": agent.online.state_dict(),
                    "target": agent.target.state_dict(),
                    "optimizer": agent.optimizer.state_dict(),
                    "history": history,
                    "best_summary": best_summary,
                    "best_metric": best_metric,
                    "config": asdict(cfg),
                }
                if cfg.save_replay_in_checkpoint:
                    payload["replay"] = replay.state_dict()
                manager.save("best_mean", payload)
                print(f"[probe] New best checkpoint saved. metric={best_metric}\n")

        if episode % cfg.checkpoint_every_episodes == 0:
            payload = {
                "episode": episode,
                "global_step": agent.global_step,
                "online": agent.online.state_dict(),
                "target": agent.target.state_dict(),
                "optimizer": agent.optimizer.state_dict(),
                "history": history,
                "best_summary": best_summary,
                "best_metric": best_metric,
                "config": asdict(cfg),
            }
            if cfg.save_replay_in_checkpoint:
                payload["replay"] = replay.state_dict()
            manager.save(f"ep_{episode:05d}", payload)

    return agent, history, best_summary

In [ ]:

# Suggested training call:
# agent, history, best_summary = train(CFG)
#
# Suggested final evaluation:
# best_ckpt = os.path.join(CFG.workdir, "checkpoints", "best_mean.pt")
# agent_loaded, ckpt = load_agent_from_checkpoint(best_ckpt, CFG)
# summary = eval_policy(agent_loaded, CFG.final_eval_seeds, CFG.final_eval_seed_start)

In [ ]:

def load_agent_from_checkpoint(ckpt_path: str, cfg: Config):
    game, state, num_actions = reset_env(cfg.seed)
    _, planes0, _, _ = observe_state(state, num_actions)
    agent = RainbowAgent(tuple(planes0.shape), num_actions, cfg)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    agent.online.load_state_dict(ckpt["online"])
    agent.target.load_state_dict(ckpt["target"])
    agent.optimizer.load_state_dict(ckpt["optimizer"])
    agent.global_step = ckpt.get("global_step", 0)
    return agent, ckpt

## Cách dùng ngắn gọn

- Muốn mạnh thật: giữ `num_episodes = 20000`
- Muốn mạnh hơn nữa: tăng lên `30000`
- Khi đánh giá: **ưu tiên `best_mean.pt`**
- Nếu cần nhắm tile cao hơn nữa:
  - tăng `probe_num_seeds`
  - tăng `buffer_size`
  - cân nhắc thêm imitation warm-start từ heuristic / expectimax